In [7]:
import numpy as np
import socket
import time
import pandas as pd


In [8]:
# Vocabolario di azioni “fisso” che Unity si aspetta (esempio one-hot a 5 lunghezze)
ACTIONS = {
    'LEFT':     [1, 0, 0, 0, 0, 0], # indice 0 => mano sinistra
    'RIGHT':    [0, 1, 0, 0, 0, 0], # indice 1 => mano destra
    'FORWARD':  [0, 0, 1, 0, 0, 0], # indice 2 => piedi
    'LINGUA':   [0, 0, 0, 1, 0, 0], # indice 3 => lingua
    'STOP':     [0, 0, 0, 0, 1, 0], # indice 4 => stop (rest)
    'PIEDE':    [0, 0, 0, 0, 0, 1], # indice 5 => piede
}

# Mappature “dataset → azione” basate su come interpreti le classi interne
DATASET_MAPPINGS = {
    'BCI2a': {
        0: ACTIONS['LEFT'],    # indice 0 => mano sinistra => LEFT
        1: ACTIONS['RIGHT'],   # indice 1 => mano destra => RIGHT
        2: ACTIONS['FORWARD'], # indice 2 => piedi => FORWARD
        3: ACTIONS['LINGUA'],    # indice 3 => lingua => Lingua
        'default': ACTIONS['STOP']
    },
    'BCI2b': {
        0: ACTIONS['LEFT'],    # indice 0 => mano sinistra => LEFT
        1: ACTIONS['RIGHT'],   # indice 1 => mano destra => RIGHT
        'default': ACTIONS['STOP']
    },
    'HGD': {
        0: ACTIONS['RIGHT'],   # indice 0 => mano destra => RIGHT
        1: ACTIONS['LEFT'],    # indice 1 => mano sinistra => LEFT
        2: ACTIONS['STOP'],    # indice 2 => rest => STOP
        3: ACTIONS['PIEDE'], # indice 3 => piede => Piede
        'default': ACTIONS['STOP']
        # … aggiungi tutte le classi che hai, mappandole su questo “vocabolario”
    }
}

def get_action_for(dataset_name, class_index):
    """
    Dato il nome del dataset (stringa) e l'indice di classe interno,
    restituisce l'array one-hot di lunghezza fissa (5 nel nostro esempio)
    che Unity sa interpretare.
    """
    mapping = DATASET_MAPPINGS[dataset_name]
    return mapping.get(class_index, mapping['default'])


In [9]:
import os
import numpy as np

def load_preds_and_trues(base_folder, dataset_name, subject_id):
    """
    Carica preds e true per un dato dataset e un dato soggetto,
    assumendo che la struttura delle cartelle sia:
    
      base_folder/
        ├─ BCI2a/
        │   ├─ BCI2a_sub1_cmds.npy
        │   ├─ BCI2a_sub1_true.npy
        │   ├─ BCI2a_sub2_cmds.npy
        │   ├─ BCI2a_sub2_true.npy
        │   └─ ...
        ├─ BCI2b/
        │   ├─ BCI2b_sub1_cmds.npy
        │   ├─ BCI2b_sub1_true.npy
        │   └─ ...
        └─ HGD/
            ├─ HGD_sub1_cmds.npy
            ├─ HGD_sub1_true.npy
            └─ ...
    
    Parametri:
      - base_folder: cartella “comandi” che contiene le sottocartelle dei dataset
                     (es. "commands" o "./commands")
      - dataset_name: nome del dataset ("BCI2a", "BCI2b" o "HGD")
      - subject_id:   numero del soggetto (intero o stringa) senza gli zeri iniziali,
                      ad esempio "1", "2", … fino a "9" o "14" per HGD.
    
    Ritorna:
      - preds: numpy array con shape (n_trials,) o (n_trials, n_classi)
      - trues: numpy array con shape (n_trials,) o (n_trials, n_classi)
    """

    # 1) Controllo che il dataset sia supportato
    if dataset_name not in {"BCI2a", "BCI2b", "HGD"}:
        raise ValueError(
            f"Dataset '{dataset_name}' non supportato. Scegli tra 'BCI2a', 'BCI2b' o 'HGD'."
        )

    # 2) Cartella specifica del dataset
    dataset_folder = os.path.join(base_folder, dataset_name)
    if not os.path.isdir(dataset_folder):
        raise FileNotFoundError(
            f"La cartella del dataset non esiste:\n  {dataset_folder}"
        )

    # 3) Costruisco i nomi dei file secondo la convenzione:
    #    <dataset_name>_sub<subject_id>_cmds.npy
    #    <dataset_name>_sub<subject_id>_true.npy
    pred_filename = f"{dataset_name}_sub{subject_id}_cmds.npy"
    true_filename = f"{dataset_name}_sub{subject_id}_true.npy"

    pred_path = os.path.join(dataset_folder, pred_filename)
    true_path = os.path.join(dataset_folder, true_filename)

    # 4) Controllo esistenza fisica dei file
    if not os.path.isfile(pred_path):
        raise FileNotFoundError(
            f"File di predizioni non trovato:\n  {pred_path}"
        )
    if not os.path.isfile(true_path):
        raise FileNotFoundError(
            f"File di true_label non trovato:\n  {true_path}"
        )

    # 5) Carico i .npy in numpy array
    preds = np.load(pred_path, allow_pickle=True)
    trues = np.load(true_path, allow_pickle=True)

    # 6) Verifico consistenza di lunghezza
    if preds.shape[0] != trues.shape[0]:
        raise ValueError(
            f"I due file hanno numeri di trial diversi:\n"
            f"  preds ha {preds.shape[0]} righe, trues ha {trues.shape[0]} righe."
        )

    return preds, trues


In [10]:
# =========================================================
# Imposta questi parametri in base alla tua struttura
# =========================================================
BASE_FOLDER  = "commands"      # cartella che contiene le sottocartelle BCI2a, BCI2b e HGD
DATASET_NAME = "BCI2a"         # o "BCI2b" o "HGD"
SUBJECT_N = "A"                # Parametro aggiunto per specificare il soggetto in modo più flessibile A o B o S
SUBJECT_ID   = "3"             # ad es. "1", "2", … (per BCI2a/BCI2b vanno da 1 a 9; HGD da 1 a 14)
RATE_HZ      = 2.0             # velocità di invio comandi (es. 2 comandi al secondo)

# Caricamento predizioni e true‐label
preds, trues = load_preds_and_trues(BASE_FOLDER, DATASET_NAME, SUBJECT_ID)
print(f"Dataset {DATASET_NAME}, Subject {SUBJECT_ID}, Numero di trial: {len(preds)}")


Dataset BCI2a, Subject 3, Numero di trial: 288


In [11]:
from pathlib import Path

#Parametri per i dati di mental work load
COLUMN_NAME = ["theta_alpha"]
csv_path = Path(f"/Users/giuseppebonomo/Desktop/RatioWaveNet/{DATASET_NAME}_IV") / "mental_work_load" / f"{SUBJECT_N}{SUBJECT_ID}" / "combined_results.csv"  # Percorso al file CSV con i dati di MWL
#/Users/giuseppebonomo/Desktop/RatioWaveNet/BCI2a_IV/mental_work_load/A3/combined_results.csv
if not csv_path.exists():
    raise FileNotFoundError(f"CSV non trovato:\n  {csv_path}")

df = pd.read_csv(csv_path)

if COLUMN_NAME[0] not in df.columns:
    raise KeyError(f"La colonna '{COLUMN_NAME[0]}' non è presente nel file!")

mwl_raw = df[COLUMN_NAME[0]].astype(float).to_numpy()

# normalizza 0‒1
mwl = (mwl_raw - mwl_raw.min()) / (mwl_raw.max() - mwl_raw.min() + 1e-9)

print(f"✔️  Caricati {len(mwl)} valori MWL da {csv_path.name}")
#stampiamo alcuni valori per debug
print(f"   min={mwl_raw.min():.3f}  max={mwl_raw.max():.3f}")
print(f"   min={mwl.min():.3f}  max={mwl.max():.3f}")
#stampa il primo valore per debug
print(f"   Primo valore MWL: {mwl[0]:.3f}")

✔️  Caricati 288 valori MWL da combined_results.csv
   min=0.097  max=3.361
   min=0.000  max=1.000
   Primo valore MWL: 1.000


In [12]:
# Parametri di connessione - Unity deve essere in ascolto sulla stessa porta
HOST = "127.0.0.1"
PORT = 5005

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.connect((HOST, PORT))
print("Connesso a Unity su", HOST, "porta", PORT)

Connesso a Unity su 127.0.0.1 porta 5005


In [16]:
# ─────────────────────────────────────────────────────────
# 3)  Inizializzo contatori e coerenza lunghezze
# ─────────────────────────────────────────────────────────
n_trials = preds.shape[0]
assert len(mwl) == n_trials, "mwl e preds devono avere lo stesso numero di trial"

delay   = 1.0 / RATE_HZ
correct = 0

# fallback: ultimo comando corretto conosciuto
first_true = trues[0]
idx0_true  = int(first_true)
last_cmd   = get_action_for(DATASET_NAME, idx0_true)

# helper per il pacchetto completo (6 one-hot + mwl)
def pack_for_unity(one_hot_cmd, mwl_val):
    vals = list(one_hot_cmd) + [round(float(mwl_val), 3)]
    return ",".join(map(str, vals)) + "\n"

# ─────────────────────────────────────────────────────────
# 4) Loop di invio
# ─────────────────────────────────────────────────────────
for i in range(n_trials):
    p, t = preds[i], trues[i]

    # 4.1) Estrazione indici predetto/vero
    idx_pred = int(p)
    idx_true = int(t)

    # 4.2) Mappatura in azioni (one-hot)
    cmd_pred = get_action_for(DATASET_NAME, idx_pred)
    cmd_true = get_action_for(DATASET_NAME, idx_true)

    # 4.3) Decisione su cosa inviare
    if cmd_pred == cmd_true:
        to_send = cmd_pred
        correct += 1
        note = "ESEGUO"
        last_cmd = cmd_true
    else:
        to_send = ACTIONS["STOP"]          # fallback
        note = "ERRATO → mando STOP"

    # 4.4) Aggiungo il MWL di questo trial e spedisco
    msg = pack_for_unity(to_send, mwl[i])  # ← qui dentro mettiamo il MWL
    sock.send(msg.encode())

    # 4.5) Log su console
    print(f"Trial {i:03d}: pred={idx_pred}->{cmd_pred}, true={idx_true}->{cmd_true}, "
          f"mwl={mwl[i]:.2f} | {note}")

    # 4.6) Attesa tra i pacchetti
    time.sleep(delay)

# ─────────────────────────────────────────────────────────
# 5) Accuracy finale e chiusura socket
# ─────────────────────────────────────────────────────────
accuracy = correct / n_trials
print(f"\nAccuracy complessiva = {accuracy:.2%} ({correct}/{n_trials})")
sock.close()


Trial 000: pred=0->[1, 0, 0, 0, 0, 0], true=0->[1, 0, 0, 0, 0, 0], mwl=1.00 | ESEGUO
Trial 001: pred=1->[0, 1, 0, 0, 0, 0], true=1->[0, 1, 0, 0, 0, 0], mwl=0.26 | ESEGUO
Trial 002: pred=1->[0, 1, 0, 0, 0, 0], true=1->[0, 1, 0, 0, 0, 0], mwl=0.22 | ESEGUO
Trial 003: pred=0->[1, 0, 0, 0, 0, 0], true=0->[1, 0, 0, 0, 0, 0], mwl=0.25 | ESEGUO
Trial 004: pred=1->[0, 1, 0, 0, 0, 0], true=1->[0, 1, 0, 0, 0, 0], mwl=0.26 | ESEGUO
Trial 005: pred=0->[1, 0, 0, 0, 0, 0], true=0->[1, 0, 0, 0, 0, 0], mwl=0.52 | ESEGUO
Trial 006: pred=1->[0, 1, 0, 0, 0, 0], true=1->[0, 1, 0, 0, 0, 0], mwl=0.30 | ESEGUO
Trial 007: pred=2->[0, 0, 1, 0, 0, 0], true=2->[0, 0, 1, 0, 0, 0], mwl=0.14 | ESEGUO
Trial 008: pred=1->[0, 1, 0, 0, 0, 0], true=1->[0, 1, 0, 0, 0, 0], mwl=0.18 | ESEGUO
Trial 009: pred=3->[0, 0, 0, 1, 0, 0], true=3->[0, 0, 0, 1, 0, 0], mwl=0.18 | ESEGUO
Trial 010: pred=0->[1, 0, 0, 0, 0, 0], true=0->[1, 0, 0, 0, 0, 0], mwl=0.21 | ESEGUO
Trial 011: pred=2->[0, 0, 1, 0, 0, 0], true=2->[0, 0, 1, 0, 0, 0]

KeyboardInterrupt: 